In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
import re
from wordcloud import WordCloud
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from bs4 import BeautifulSoup
import requests
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import random

In [2]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
def scrape_amazon_reviews_manual(reviews_page_url = "https://www.amazon.in/product-reviews/B0FCG45VHF/ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews", max_pages=5):
    """
    Direct scraper for Amazon reviews page URL

    To get the reviews page URL:
    1. Go to product page
    2. Click "See all reviews"
    3. Copy the URL from browser
    4. Paste it here

    Parameters:
    - reviews_page_url: Direct URL to reviews page (starts with https://www.amazon.in/product-reviews/)
    - max_pages: Number of pages to scrape
    """
    reviews_data = {
        'review_id': [],
        'product_name': [],
        'rating': [],
        'review_title': [],
        'review_text': [],
        'reviewer_name': [],
        'review_date': [],
        'verified_purchase': []
    }

    print(f"🔍 Starting manual Amazon scraper...")
    print(f"📄 URL: {reviews_page_url}")

    chrome_options = Options()
    # chrome_options.add_argument('--headless')  # Comment out to see browser
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    except Exception as e:
        print(f"❌ Error initializing Chrome driver: {e}")
        return pd.DataFrame()

    try:
        driver.get(reviews_page_url)
        time.sleep(4)

        # Get product name from reviews page
        try:
            product_name = driver.find_element(By.CSS_SELECTOR, "a[data-hook='product-link']").text.strip()
            print(f"📦 Product: {product_name}")
        except:
            product_name = "Unknown Product"

        for page in range(max_pages):
            print(f"📖 Scraping page {page + 1}/{max_pages}...")
            time.sleep(2)

            # Scroll to load all reviews
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, 'lxml')
            reviews = soup.find_all('div', {'data-hook': 'review'})

            print(f"   Found {len(reviews)} review elements on this page")

            for idx, review in enumerate(reviews):
                try:
                    # Rating
                    rating_elem = review.find('i', {'data-hook': 'review-star-rating'})
                    if not rating_elem:
                        rating_elem = review.find('i', class_=lambda x: x and 'a-star' in str(x))
                    rating = None
                    if rating_elem:
                        rating_text = rating_elem.text.strip()
                        try:
                            rating = float(rating_text.split()[0])
                        except:
                            pass

                    # Title
                    title_elem = review.find('a', {'data-hook': 'review-title'})
                    if not title_elem:
                        title_elem = review.find('span', {'data-hook': 'review-title'})
                    title = title_elem.text.strip() if title_elem else ""

                    # Review text
                    text_elem = review.find('span', {'data-hook': 'review-body'})
                    text = text_elem.text.strip() if text_elem else ""

                    # Reviewer
                    name_elem = review.find('span', {'class': 'a-profile-name'})
                    name = name_elem.text.strip() if name_elem else "Anonymous"

                    # Date
                    date_elem = review.find('span', {'data-hook': 'review-date'})
                    date = date_elem.text.strip() if date_elem else ""

                    # Verified
                    verified = bool(review.find('span', {'data-hook': 'avp-badge'}))

                    if text and len(text) > 10:
                        reviews_data['review_id'].append(f"AMZ_{page}_{idx}")
                        reviews_data['product_name'].append(product_name)
                        reviews_data['rating'].append(rating)
                        reviews_data['review_title'].append(title)
                        reviews_data['review_text'].append(text)
                        reviews_data['reviewer_name'].append(name)
                        reviews_data['review_date'].append(date)
                        reviews_data['verified_purchase'].append(verified)
                except:
                    continue

            print(f"   ✓ Extracted {len([r for r in reviews_data['review_id'] if r.startswith(f'AMZ_{page}')])} reviews")

            # Next page
            if page < max_pages - 1:
                try:
                    next_button = driver.find_element(By.CSS_SELECTOR, "li.a-last a")
                    driver.execute_script("arguments[0].click();", next_button)
                    time.sleep(3)
                except:
                    print("   ℹ️ No more pages")
                    break

        print(f"\n✅ Total scraped: {len(reviews_data['review_id'])} reviews")

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        driver.quit()

    return pd.DataFrame(reviews_data)

In [4]:
def scrape_amazon_reviews(product_url, max_pages=5):
    """
    Scrape reviews from Amazon product page

    Parameters:
    - product_url: URL of the Amazon product
    - max_pages: Number of review pages to scrape (default: 5)

    Returns:
    - DataFrame with reviews
    """
    reviews_data = {
        'review_id': [],
        'product_name': [],
        'rating': [],
        'review_title': [],
        'review_text': [],
        'reviewer_name': [],
        'review_date': [],
        'verified_purchase': []
    }

    print(f"🔍 Starting Amazon scraper...")

    # Setup Chrome options
    chrome_options = Options()
    # chrome_options.add_argument('--headless')  # Disabled for debugging
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    # Initialize driver
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    except Exception as e:
        print(f"❌ Error initializing Chrome driver: {e}")
        return pd.DataFrame()

    try:
        # Navigate to product page
        print(f"📄 Loading product page...")
        driver.get(product_url)
        time.sleep(3)

        # Get product name
        try:
            product_name = driver.find_element(By.ID, "productTitle").text.strip()
            print(f"📦 Product: {product_name}")
        except:
            try:
                product_name = driver.find_element(By.CSS_SELECTOR, "span#productTitle").text.strip()
            except:
                product_name = "Unknown Product"

        # Scroll to load reviews section
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(2)

        # Click "See all reviews" button - try multiple methods
        reviews_url = None
        try:
            # Method 1: Find by data-hook
            see_all_reviews = driver.find_element(By.CSS_SELECTOR, "a[data-hook='see-all-reviews-link-foot']")
            reviews_url = see_all_reviews.get_attribute('href')
        except:
            try:
                # Method 2: Find by partial text
                see_all = driver.find_element(By.XPATH, "//a[contains(text(), 'See all reviews') or contains(text(), 'customer review')]")
                reviews_url = see_all.get_attribute('href')
            except:
                try:
                    # Method 3: Find reviews link in the page
                    soup_temp = BeautifulSoup(driver.page_source, 'lxml')
                    reviews_link = soup_temp.find('a', string=lambda x: x and 'review' in x.lower())
                    if reviews_link and reviews_link.get('href'):
                        reviews_url = reviews_link['href']
                        if not reviews_url.startswith('http'):
                            reviews_url = 'https://www.amazon.in' + reviews_url
                except:
                    pass

        if reviews_url:
            print(f"📄 Navigating to reviews page...")
            driver.get(reviews_url)
            time.sleep(3)
        else:
            print("⚠️ Trying to extract reviews from product page directly...")
            # Continue with current page

        # Scrape multiple pages
        for page in range(max_pages):
            print(f"📖 Scraping page {page + 1}/{max_pages}...")

            try:
                # Wait for reviews to load - multiple possible selectors
                try:
                    WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-hook='review']"))
                    )
                except:
                    try:
                        WebDriverWait(driver, 5).until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, "div.review"))
                        )
                    except:
                        print(f"⚠️ Trying alternative review selectors...")
            except:
                print(f"⚠️ No reviews found on page {page + 1}")
                if page == 0:
                    # Try one more time with different approach
                    print("🔄 Attempting alternative extraction method...")
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(3)
                else:
                    break

            # Parse with BeautifulSoup
            soup = BeautifulSoup(driver.page_source, 'lxml')

            # Try multiple selectors for reviews
            reviews = soup.find_all('div', {'data-hook': 'review'})

            if not reviews:
                reviews = soup.find_all('div', class_=lambda x: x and 'review' in str(x).lower() and 'card' in str(x).lower())

            if not reviews:
                reviews = soup.find_all('div', id=lambda x: x and 'customer_review' in str(x).lower())

            if not reviews:
                # Last resort - look for any div containing review-like content
                all_divs = soup.find_all('div')
                reviews = [d for d in all_divs if d.find('span', {'data-hook': 'review-body'}) or
                          (d.find('i', class_=lambda x: x and 'review-rating' in str(x)))]

            if not reviews:
                print(f"   ❌ Could not find any reviews on page {page + 1}")
                if page == 0:
                    print("\n💡 TIP: This might be due to:")
                    print("   1. Product has no reviews yet")
                    print("   2. Reviews are behind a CAPTCHA/login")
                    print("   3. Amazon's HTML structure changed")
                    print("   4. Try using '--disable-blink-features=AutomationControlled' or run without headless mode")
                break

            for idx, review in enumerate(reviews):
                try:
                    # Extract rating
                    rating_elem = review.find('i', {'data-hook': 'review-star-rating'})
                    if not rating_elem:
                        rating_elem = review.find('i', class_=lambda x: x and 'review-rating' in str(x))
                    rating = float(rating_elem.text.split()[0]) if rating_elem else None

                    # Extract review title
                    title_elem = review.find('a', {'data-hook': 'review-title'})
                    if not title_elem:
                        title_elem = review.find('span', {'data-hook': 'review-title'})
                    title = title_elem.text.strip() if title_elem else ""

                    # Extract review text
                    text_elem = review.find('span', {'data-hook': 'review-body'})
                    text = text_elem.text.strip() if text_elem else ""

                    # Extract reviewer name
                    name_elem = review.find('span', {'class': 'a-profile-name'})
                    name = name_elem.text.strip() if name_elem else "Anonymous"

                    # Extract review date
                    date_elem = review.find('span', {'data-hook': 'review-date'})
                    date = date_elem.text.strip() if date_elem else ""

                    # Check if verified purchase
                    verified = bool(review.find('span', {'data-hook': 'avp-badge'}))

                    # Only add if review has text
                    if text and len(text) > 10:
                        reviews_data['review_id'].append(f"AMZ_{page}_{idx}")
                        reviews_data['product_name'].append(product_name)
                        reviews_data['rating'].append(rating)
                        reviews_data['review_title'].append(title)
                        reviews_data['review_text'].append(text)
                        reviews_data['reviewer_name'].append(name)
                        reviews_data['review_date'].append(date)
                        reviews_data['verified_purchase'].append(verified)

                except Exception as e:
                    continue

            print(f"   ✓ Found {len([r for r in reviews_data['review_id'] if r.startswith(f'AMZ_{page}')])} reviews on this page")

            # Try to go to next page
            if page < max_pages - 1:
                try:
                    next_button = driver.find_element(By.CSS_SELECTOR, "li.a-last a")
                    next_button.click()
                    time.sleep(random.uniform(2, 4))
                except:
                    print("   ℹ️ No more pages available")
                    break

        total_reviews = len(reviews_data['review_id'])
        print(f"\n✅ Successfully scraped {total_reviews} Amazon reviews!")

    except Exception as e:
        print(f"❌ Error during scraping: {e}")

    finally:
        driver.quit()

    return pd.DataFrame(reviews_data)

In [5]:
def scrape_flipkart_reviews(product_url, max_pages=5):
    """
    Scrape reviews from Flipkart product page

    Parameters:
    - product_url: URL of the Flipkart product
    - max_pages: Number of review pages to scrape (default: 5)

    Returns:
    - DataFrame with reviews
    """
    reviews_data = {
        'review_id': [],
        'product_name': [],
        'rating': [],
        'review_title': [],
        'review_text': [],
        'reviewer_name': [],
        'review_date': [],
        'verified_purchase': []
    }

    print(f"🔍 Starting Flipkart scraper...")

    # Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    # Initialize driver
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    except Exception as e:
        print(f"❌ Error initializing Chrome driver: {e}")
        return pd.DataFrame()

    try:
        # Navigate to product page
        print(f"📄 Loading product page...")
        driver.get(product_url)
        time.sleep(3)

        # Get product name
        try:
            product_name = driver.find_element(By.CSS_SELECTOR, "span.VU-ZEz").text.strip()
            print(f"📦 Product: {product_name}")
        except:
            try:
                product_name = driver.find_element(By.CSS_SELECTOR, "span.B_NuCI").text.strip()
            except:
                product_name = "Unknown Product"

        # Scroll to reviews section
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(2)

        # Try to find and click "All reviews" or rating summary
        try:
            all_reviews = driver.find_element(By.XPATH, "//div[contains(@class, 'col') and contains(@class, 'reviews')]")
            driver.execute_script("arguments[0].scrollIntoView();", all_reviews)
            time.sleep(1)
        except:
            pass

        # Scrape multiple pages
        for page in range(max_pages):
            print(f"📖 Scraping page {page + 1}/{max_pages}...")

            time.sleep(2)

            # Parse with BeautifulSoup
            soup = BeautifulSoup(driver.page_source, 'lxml')

            # Find review containers (multiple possible selectors)
            reviews = soup.find_all('div', class_=lambda x: x and ('_1AtVbE' in str(x) or 'col-12-12' in str(x)))

            if not reviews:
                reviews = soup.find_all('div', class_=lambda x: x and 'review' in str(x).lower())

            page_count = 0
            for idx, review in enumerate(reviews):
                try:
                    # Extract rating
                    rating = None
                    rating_elem = review.find('div', class_=lambda x: x and ('hGSR34' in str(x) or 'XQDdHH' in str(x)))
                    if rating_elem:
                        rating_text = rating_elem.text.strip()
                        try:
                            rating = float(rating_text.split()[0])
                        except:
                            pass

                    # Extract review title/summary
                    title = ""
                    title_elem = review.find('p', class_=lambda x: x and '_2-N8zT' in str(x))
                    if title_elem:
                        title = title_elem.text.strip()

                    # Extract review text
                    text = ""
                    text_elem = review.find('div', class_=lambda x: x and 't-ZTKy' in str(x))
                    if not text_elem:
                        text_elem = review.find('div', class_=lambda x: x and 'review' in str(x).lower())
                    if text_elem:
                        text = text_elem.text.strip()

                    # Extract reviewer name
                    name = "Anonymous"
                    name_elem = review.find('p', class_=lambda x: x and '_2sc7ZR' in str(x))
                    if name_elem:
                        name_parts = name_elem.text.strip().split(',')
                        name = name_parts[0] if name_parts else "Anonymous"

                    # Extract date
                    date = ""
                    if name_elem and ',' in name_elem.text:
                        date_parts = name_elem.text.split(',')
                        date = date_parts[-1].strip() if len(date_parts) > 1 else ""

                    # Check verified purchase
                    verified = bool(review.find(string=lambda x: x and 'certified buyer' in str(x).lower()))

                    # Only add if review has meaningful text
                    if text and len(text) > 10:
                        reviews_data['review_id'].append(f"FK_{page}_{idx}")
                        reviews_data['product_name'].append(product_name)
                        reviews_data['rating'].append(rating)
                        reviews_data['review_title'].append(title)
                        reviews_data['review_text'].append(text)
                        reviews_data['reviewer_name'].append(name)
                        reviews_data['review_date'].append(date)
                        reviews_data['verified_purchase'].append(verified)
                        page_count += 1

                except Exception as e:
                    continue

            print(f"   ✓ Found {page_count} reviews on this page")

            # Try to go to next page
            if page < max_pages - 1:
                try:
                    next_button = driver.find_element(By.XPATH, "//a[contains(@class, '_9QVEpD') or contains(text(), 'Next')]")
                    next_button.click()
                    time.sleep(random.uniform(2, 4))
                except:
                    print("   ℹ️ No more pages available")
                    break

        total_reviews = len(reviews_data['review_id'])
        print(f"\n✅ Successfully scraped {total_reviews} Flipkart reviews!")

    except Exception as e:
        print(f"❌ Error during scraping: {e}")

    finally:
        driver.quit()

    return pd.DataFrame(reviews_data)

In [6]:
def analyze_product_from_url(url, max_pages=5):
    """
    Main function: Scrape and analyze product reviews from URL

    Parameters:
    - url: Product URL from Amazon or Flipkart
    - max_pages: Number of review pages to scrape (default: 5)

    Returns:
    - DataFrame with reviews and sentiment analysis
    """
    print("=" * 70)
    print("🚀 PRODUCT REVIEW SENTIMENT ANALYSIS")
    print("=" * 70)

    # Auto-detect platform
    if 'amazon' in url.lower():
        platform = 'amazon'
        print(f"🛍️ Platform detected: Amazon")
        df = scrape_amazon_reviews(url, max_pages)
    elif 'flipkart' in url.lower():
        platform = 'flipkart'
        print(f"🛍️ Platform detected: Flipkart")
        df = scrape_flipkart_reviews(url, max_pages)
    else:
        print("❌ Could not detect platform. URL must be from Amazon or Flipkart.")
        return pd.DataFrame()

    if df.empty:
        print("❌ No reviews found. Please check the URL and try again.")
        return pd.DataFrame()

    print(f"\n{'=' * 70}")
    print("🔬 PERFORMING SENTIMENT ANALYSIS")
    print("=" * 70)

    # Perform sentiment analysis
    df = perform_sentiment_analysis(df)

    # Generate visualizations
    print("\n📊 Generating visualizations...")
    visualize_sentiment(df)

    # Generate summary report
    print("\n📋 GENERATING SUMMARY REPORT")
    print("=" * 70)
    generate_summary_report(df)

    return df

In [7]:
def clean_text(text):
    """Clean and preprocess review text"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join(text.split())
    return text

def get_sentiment(text):
    """Analyze sentiment using TextBlob"""
    if not text or len(text) < 3:
        return pd.Series([0, 0, 'Neutral'])

    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    if polarity > 0.1:
        sentiment = 'Positive'
    elif polarity < -0.1:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'

    return pd.Series([polarity, subjectivity, sentiment])

def perform_sentiment_analysis(df):
    """Perform complete sentiment analysis on reviews"""
    print("🧹 Cleaning text...")
    df['cleaned_text'] = df['review_text'].apply(clean_text)

    print("🤖 Analyzing sentiment...")
    df[['polarity', 'subjectivity', 'sentiment']] = df['cleaned_text'].apply(get_sentiment)

    print(f"✅ Sentiment analysis complete!")
    return df

In [8]:
def visualize_sentiment(df):
    """Generate all visualizations for sentiment analysis"""

    # 1. Overall Sentiment Distribution
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    sentiment_counts = df['sentiment'].value_counts()
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']

    # Pie chart
    sentiment_counts.plot(kind='pie', ax=axes[0, 0], autopct='%1.1f%%',
                          colors=colors, startangle=90)
    axes[0, 0].set_title('Overall Sentiment Distribution', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('')

    # Bar chart
    sentiment_counts.plot(kind='bar', ax=axes[0, 1], color=colors)
    axes[0, 1].set_title('Sentiment Frequency', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Sentiment', fontsize=12)
    axes[0, 1].set_ylabel('Count', fontsize=12)
    axes[0, 1].tick_params(axis='x', rotation=0)

    # Polarity distribution
    axes[1, 0].hist(df['polarity'], bins=20, color='skyblue', edgecolor='black')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral')
    axes[1, 0].set_title('Polarity Score Distribution', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Polarity Score', fontsize=12)
    axes[1, 0].set_ylabel('Frequency', fontsize=12)
    axes[1, 0].legend()

    # Rating vs Sentiment
    if 'rating' in df.columns and not df['rating'].isna().all():
        rating_sentiment = pd.crosstab(df['rating'], df['sentiment'])
        rating_sentiment.plot(kind='bar', ax=axes[1, 1], color=colors, stacked=False)
        axes[1, 1].set_title('Sentiment by Rating', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Rating', fontsize=12)
        axes[1, 1].set_ylabel('Count', fontsize=12)
        axes[1, 1].legend(title='Sentiment')
        axes[1, 1].tick_params(axis='x', rotation=0)
    else:
        axes[1, 1].text(0.5, 0.5, 'Rating data not available',
                        ha='center', va='center', fontsize=12)
        axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

    # 2. Word Clouds
    print("\n📝 Generating word clouds...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for idx, sentiment in enumerate(['Positive', 'Negative', 'Neutral']):
        if sentiment in df['sentiment'].values:
            text = ' '.join(df[df['sentiment'] == sentiment]['cleaned_text'])
            if text:
                wordcloud = WordCloud(width=400, height=300, background_color='white',
                                     colormap='viridis', max_words=50).generate(text)
                axes[idx].imshow(wordcloud, interpolation='bilinear')
                axes[idx].set_title(f'{sentiment} Reviews', fontsize=14, fontweight='bold')
            else:
                axes[idx].text(0.5, 0.5, f'No {sentiment} reviews',
                              ha='center', va='center')
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [9]:
def generate_summary_report(df):
    """Generate comprehensive summary report"""

    product_name = df['product_name'].iloc[0] if not df.empty else "Unknown"
    total_reviews = len(df)

    print(f"\n📦 PRODUCT: {product_name}")
    print(f"📊 Total Reviews Analyzed: {total_reviews}")
    print("\n" + "─" * 70)

    # Sentiment breakdown
    sentiment_counts = df['sentiment'].value_counts()
    print("\n🎯 SENTIMENT BREAKDOWN:")
    for sentiment in ['Positive', 'Negative', 'Neutral']:
        count = sentiment_counts.get(sentiment, 0)
        percentage = (count / total_reviews * 100) if total_reviews > 0 else 0
        emoji = "😊" if sentiment == "Positive" else "😞" if sentiment == "Negative" else "😐"
        print(f"   {emoji} {sentiment:12s}: {count:4d} reviews ({percentage:5.1f}%)")

    # Average scores
    print("\n📈 AVERAGE SCORES:")
    print(f"   Polarity:     {df['polarity'].mean():+.3f} (range: -1 to +1)")
    print(f"   Subjectivity: {df['subjectivity'].mean():.3f} (range: 0 to 1)")

    if 'rating' in df.columns and not df['rating'].isna().all():
        print(f"   Star Rating:  {df['rating'].mean():.2f} / 5.0")

    # Verified purchases
    if 'verified_purchase' in df.columns:
        verified_count = df['verified_purchase'].sum()
        verified_pct = (verified_count / total_reviews * 100) if total_reviews > 0 else 0
        print(f"\n✅ Verified Purchases: {verified_count} ({verified_pct:.1f}%)")

    # Most common words
    print("\n📝 TOP KEYWORDS BY SENTIMENT:")

    for sentiment in ['Positive', 'Negative']:
        if sentiment in df['sentiment'].values:
            text = ' '.join(df[df['sentiment'] == sentiment]['cleaned_text'])
            words = text.split()
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
                         'for', 'of', 'with', 'is', 'it', 'this', 'that', 'very', 'so',
                         'my', 'i', 'im', 'ive', 'dont', 'didnt', 'was', 'were'}
            filtered_words = [w for w in words if w not in stop_words and len(w) > 2]

            if filtered_words:
                top_words = Counter(filtered_words).most_common(5)
                emoji = "😊" if sentiment == "Positive" else "😞"
                print(f"\n   {emoji} {sentiment}:")
                for word, count in top_words:
                    print(f"      • {word} ({count})")

    # Overall recommendation
    positive_pct = (sentiment_counts.get('Positive', 0) / total_reviews * 100) if total_reviews > 0 else 0

    print("\n" + "═" * 70)
    print("💡 OVERALL ASSESSMENT:")

    if positive_pct >= 70:
        print("   ⭐⭐⭐⭐⭐ HIGHLY RECOMMENDED")
        print("   Most customers are satisfied with this product!")
    elif positive_pct >= 50:
        print("   ⭐⭐⭐⭐ RECOMMENDED")
        print("   Generally positive reviews with some concerns.")
    elif positive_pct >= 30:
        print("   ⭐⭐⭐ MIXED REVIEWS")
        print("   Product has both strengths and weaknesses.")
    else:
        print("   ⭐⭐ CAUTION ADVISED")
        print("   Many negative reviews. Consider alternatives.")

    print("═" * 70)

    # Save results
    try:
        output_file = 'sentiment_analysis_results.csv'
        df.to_csv(output_file, index=False)
        print(f"\n💾 Results saved to: {output_file}")
    except:
        print("\n⚠️ Could not save results to file")

In [10]:
product_url = "YOUR_PRODUCT_URL_HERE"

# Number of review pages to scrape (more pages = more reviews but slower)
max_pages = 5

# ============================================================================
# STEP 2: RUN THIS CELL TO START ANALYSIS
# ============================================================================

if product_url != "YOUR_PRODUCT_URL_HERE":
    df = analyze_product_from_url(product_url, max_pages=max_pages)

    if not df.empty:
        print("\n✅ Analysis complete! Check the visualizations above.")
        print(f"📊 DataFrame 'df' contains {len(df)} reviews with sentiment scores.")
        print("\n💡 You can now explore the data further:")
        print("   - df.head() - View first few reviews")
        print("   - df['sentiment'].value_counts() - Count by sentiment")
        print("   - df[df['sentiment']=='Negative'] - View only negative reviews")
else:
    print("⚠️ Please replace 'YOUR_PRODUCT_URL_HERE' with an actual product URL")
    print("\nExample URLs:")
    print("Amazon: https://www.amazon.in/dp/B08XXXXXX")
    print("Flipkart: https://www.flipkart.com/product/p/itmXXXXXX")

⚠️ Please replace 'YOUR_PRODUCT_URL_HERE' with an actual product URL

Example URLs:
Amazon: https://www.amazon.in/dp/B08XXXXXX
Flipkart: https://www.flipkart.com/product/p/itmXXXXXX
